# Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import LocallyLinearEmbedding

## Load and inspect data

In [ ]:
data_location = Path().cwd().parent/"data"

In [ ]:
# load dataset and remove the last empty columns
df = pd.read_csv(data_location / "Data_for_Task_1.csv")
df = df.drop("Unnamed: 32", axis=1)
df.columns = [col.replace(" ", "_") for col in df.columns]
df

In [ ]:
# 30 float features, 1 str (the target) and one 1 int (the id)
df.info()

In [ ]:
# checking for duplicated entries
# There are 569 different records - no duplicates
df["id"].nunique()

In [ ]:
df.duplicated().sum()

In [ ]:
# no missing data
df.isnull().sum()

In [ ]:
diagnosis_count = df["diagnosis"].value_counts(ascending=True)
diagnosis_count

In [ ]:
diagnosis_proportion = diagnosis_count / len(df)
diagnosis_proportion

## Target proportions

In [ ]:
# 63% of the samples represent benign tumors
# 37% of the samples represent malignant tumors
fig, ax = plt.subplots()
sns.countplot(data=df, x="diagnosis", stat="percent", palette="tab10",  hue="diagnosis", ax=ax)
ax.spines[["top", "right", "bottom", "left"]].set_visible(False)
ax.grid(True, linewidth=0.5, linestyle=':', axis="y")
for x, proportion in enumerate(diagnosis_proportion):
    ax.text(x-0.07, proportion*101, f"{proportion*100:.1f}%")

several features show a clear linear relationship, e.g. radius_mean and perimenter_mean  
some other show non-linear relationship such as area_mean and radius_mean, which is expected since $Area = \pi \times r^{2}$  
Some degree of linear relationships are also observed between the _mean and _worst features. This is also expected, for cases such as radius_mean & perimeter_worst  

## Dimensionality reduction

In [ ]:
X = df.drop(["id", "diagnosis"], axis=1)
X.head()

In [ ]:
y = df["diagnosis"]
y.head()

In [ ]:
pca = PCA(n_components=2)
X_transformed = pca.fit_transform(X)
df_proj = pd.DataFrame(np.c_[X_transformed, y.values], columns=["feature_1", "feature_2", "target"])

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
sns.scatterplot(df_proj, x="feature_1", y="feature_2", hue="target", ax=ax)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, linewidth=0.5, linestyle=":")
ax.set_title("PCA")

In [ ]:
pca_3d = PCA(n_components=3)
X_transformed_3d = pca_3d.fit_transform(X)
df_proj_3d = pd.DataFrame(np.c_[X_transformed_3d, y.values], columns=["feature_1", "feature_2","feature_3", "target"])

In [ ]:
m = df_proj_3d[df_proj_3d["target"]=="M"]
b = df_proj_3d[df_proj_3d["target"]=="B"]

In [ ]:
b["feature_1"].astype(float)

In [ ]:
b.info()

In [ ]:
fig, ax = plt.subplots(figsize=(6,6), subplot_kw={"projection":"3d"})
ax.scatter(m["feature_1"].astype(float), m["feature_2"].astype(float), m["feature_3"].astype(float))
ax.scatter(b["feature_1"].astype(float), b["feature_2"].astype(float), b["feature_3"].astype(float))

In [ ]:
df_cov = df.drop("id", axis=1)
df_cov = df_cov.groupby("diagnosis").cov()

In [ ]:
df_corr = df.drop("id", axis=1)
df_corr = df_corr.groupby("diagnosis").corr()

In [ ]:
df_cov

In [ ]:
df_corr

In [ ]:
cov_mask = np.tril(np.ones_like(df_cov.loc["B",:], dtype=bool),k=-1)

In [ ]:
fig, axs = plt.subplots(figsize=(50,50), nrows=2, ncols=1)
sns.heatmap(
    df_cov.loc["B",:],
    mask=cov_mask,
    cmap="vanimo",
    annot=True,
    square=True,
    ax=axs[0]
)
sns.heatmap(
    df_cov.loc["M",:],
    mask=cov_mask,
    cmap="vanimo",
    annot=True,
    square=True,
    ax=axs[1]
)
for ax in axs:
    ax.xaxis.tick_top()
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=14)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)
    ax.tick_params(axis="both", length=0)
    ax.tick_params(axis="x", pad=10)

## Scatter plot and correlations

In [ ]:
df.columns

In [ ]:
# check mean and worse features together
pairplot = sns.pairplot(df.drop(["id"], axis=1), hue="diagnosis")
for ax in pairplot.axes.flatten():
    ax.set_xlabel(ax.get_xlabel(), fontsize=18)
    ax.set_ylabel(ax.get_ylabel(), fontsize=18)

In [ ]:
# check mean and worse features together
mean_and_worst_features = [col for col in df.columns if col.endswith(("_mean", "_worst"))]
sns.pairplot(df, hue="diagnosis", vars=mean_and_worst_features)

In [ ]:
mean_features = [col for col in df.columns if col.endswith("_mean")]
sns.pairplot(df, hue="diagnosis", vars=mean_features)

In [ ]:
worst_features = [col for col in df.columns if col.endswith("_worst")]
sns.pairplot(df, hue="diagnosis", vars=worst_features)

In [ ]:
se_features = [col for col in df.columns if col.endswith("_se")]
sns.pairplot(df, hue="diagnosis", vars=se_features)

In [ ]:
# checking correlations between features
mean_and_worst_corr = df.loc[:, mean_and_worst_features].corr()

In [ ]:
mean_and_worst_mask = np.tril(np.ones_like(mean_and_worst_corr, dtype=bool),k=-1)

In [ ]:
fig, ax = plt.subplots(figsize=(15,15))
sns.heatmap(
    mean_and_worst_corr,
    mask=mean_and_worst_mask,
    vmin=-1,
    vmax=1,
    center=0,
    cmap="vanimo",
    annot=True,
    square=True,
    ax=ax
)
ax.xaxis.tick_top()
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=14)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)
ax.tick_params(axis="both", length=0)
ax.tick_params(axis="x", pad=10)


## Distributions of the features

In [ ]:
features = df.columns.to_list()[2:]
print(features)

In [ ]:
fig, axs = plt.subplots(figsize=(20,20), nrows=6, ncols=5)
for ax, feature in zip(axs.flatten(), features):
    sns.histplot(
        df,
        x=feature,
        hue="diagnosis",
        stat="density",
        kde=True,
        ax=ax
    )
    ax.grid(True, linewidth=0.5, linestyle=":", axis="y")
    ax.spines[["right","top"]].set_visible(False)
fig.tight_layout()

## Outliers

The dataset present some outliers in all categories. Sometimes more than others.  
Some features show a clear difference between the two classes and other show some strong overlapping. These should be checked more closely with ANOVA or a welch's test to see if they can be considered different and help discrimite between classes.

On the other hand, the outliers per feature should be checked, to see what the whole spread of the data per feature is.

In [ ]:
fig, axs = plt.subplots(figsize=(20,20), nrows=6, ncols=5)
for ax, feature in zip(axs.flatten(), features):
    sns.boxplot(
        df,
        x="diagnosis",
        y=feature,
        hue="diagnosis",
        ax=ax
    )
    ax.grid(True, linewidth=0.5, linestyle=":", axis="y")
    ax.spines[["right","top"]].set_visible(False)
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(figsize=(20,20), nrows=6, ncols=5)
for ax, feature in zip(axs.flatten(), features):
    sns.boxplot(
        df,
        y=feature,
        ax=ax
    )
    ax.grid(True, linewidth=0.5, linestyle=":", axis="y")
    ax.spines[["right","top"]].set_visible(False)
fig.tight_layout()

In [ ]:
Q1 = df[features].quantile(.25)
Q3 = df[features].quantile(.75)
IQR = Q3 - Q1

In [ ]:
IQR.head()

In [ ]:
lower_bound = Q1 - (1.5*IQR)
lower_bound.head()

In [ ]:
upper_bound = Q3 + (1.5*IQR)
upper_bound.head()

In [ ]:
outlier_count_dict = {}
for feature in features:
    n_outliers = ((df[feature] < lower_bound[feature]) | (df[feature] > upper_bound[feature])).sum()
    outlier_count_dict[feature] = n_outliers

In [ ]:
outlier_series = pd.Series(outlier_count_dict).sort_values(ascending=False)
outlier_series

In [ ]:
# TODO: improve figure
sns.barplot(outlier_series, orient="h")

The features with the most outliers are mostly the _se features, particularly _area_se_. 
The boxplots show that the area_se between diagnosis does not strongly overlap and also interestingly the spread of the _M_ samples is larger.

Two approaches could be interesting, drop 25 samples with the largest area_se to reduce the number of outliers or drop the feature completely.  
Relevance of this feature should be determined to decide how to proceed.

In [ ]:
# Dataset for baseline models
df.to_csv(data_location / "baseline_data.csv", index=False)